In [112]:
import pandas as pd
import matplotlib.pyplot as plt
import folium
import pandas as pd
import unidecode
from rapidfuzz import process, fuzz
from streamlit_folium import st_folium
import numpy as np

## Consolidating the Data

In [113]:
RAW = "../data/raw/"  # adjust to your local path if needed
 
FATOR_SIN = 0.060  # tCO2/MWh — fator de emissão do SIN (EPE 2023)
FONTES_RENOVAVEIS = ["EOL", "UFV", "UHE", "PCH", "CGH", "UTEBiomassa"]
 
# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────
 
def norm_municipio(s):
    return (
        s.astype(str)
         .str.split(" - ").str[0]
         .str.strip()
         .str.lower()
         .apply(unidecode.unidecode)
         .str.upper()
    )
 
def norm_uf(s):
    return s.astype(str).str.strip().str.upper()
 
def fix_decimal(s):
    return pd.to_numeric(
        s.astype(str).str.replace(",", ".", regex=False).str.strip(),
        errors="coerce"
    )
 
 
# ─────────────────────────────────────────────
# 1. CORE_DF — municipality index with coords
# ─────────────────────────────────────────────
 
siga_raw = pd.read_csv(
    RAW + "ANEEL_SIGA_empreendimentos-geracao.csv",
    sep=";", encoding="latin1"
)
 
siga_raw["municipio"]      = norm_municipio(siga_raw["DscMuninicpios"])
siga_raw["uf"]             = norm_uf(siga_raw["SigUFPrincipal"])
siga_raw["lat"]            = fix_decimal(siga_raw["NumCoordNEmpreendimento"])
siga_raw["lon"]            = fix_decimal(siga_raw["NumCoordEEmpreendimento"])
siga_raw["mw_fiscalizado"] = fix_decimal(siga_raw["MdaPotenciaFiscalizadaKw"])
siga_raw["mw_outorgado"]   = fix_decimal(siga_raw["MdaPotenciaOutorgadaKw"])
siga_raw["tipo"]           = siga_raw["SigTipoGeracao"].str.strip()
siga_raw["fase"]           = siga_raw["DscFaseUsina"].str.strip()
 
core_df = (
    siga_raw.groupby(["municipio", "uf"])
            .agg(lat=("lat", "mean"), lon=("lon", "mean"))
            .reset_index()
            .dropna(subset=["lat", "lon"])
)
 
print(f"core_df: {core_df.shape}")
 
 
# ─────────────────────────────────────────────
# 2. ECONOMIC VARIABLES  (prefix: eco_)
# ─────────────────────────────────────────────
 
# --- 2a. SIGA: MW installed, idle, pipeline ---
siga_op = siga_raw[siga_raw["fase"] == "Operação"]
 
eco_siga = (
    siga_raw.groupby(["municipio", "uf"])
            .agg(
                eco_mw_instalado   =("mw_fiscalizado", "sum"),
                eco_mw_outorgado   =("mw_outorgado",   "sum"),
                eco_n_plantas      =("tipo",            "count"),
            )
            .reset_index()
)
eco_siga["eco_mw_ocioso"] = (
    eco_siga["eco_mw_outorgado"] - eco_siga["eco_mw_instalado"]
).clip(lower=0)
 
# share of renewable MW
ren = (
    siga_raw[siga_raw["tipo"].isin(FONTES_RENOVAVEIS)]
    .groupby(["municipio", "uf"])["mw_fiscalizado"].sum()
    .reset_index()
    .rename(columns={"mw_fiscalizado": "mw_renovavel"})
)
eco_siga = eco_siga.merge(ren, on=["municipio", "uf"], how="left")
eco_siga["eco_share_renovavel"] = (
    eco_siga["mw_renovavel"] / eco_siga["eco_mw_instalado"].replace(0, pd.NA)
).fillna(0).clip(0, 1)
eco_siga = eco_siga.drop(columns=["mw_renovavel"])
 
# --- 2b. SIGET linhas: transmission infrastructure per UF ---
linhas = pd.read_csv(
    RAW + "ANEEL_SIGET_linhas-transmissao.csv",
    sep=";", encoding="latin1"
)
linhas["uf"] = norm_uf(linhas["SigUFSubestacaoOrigem"])
linhas["tensao"] = fix_decimal(linhas["NumTensaoBaseLinhaTransm"])
 
eco_linhas = (
    linhas.groupby("uf")
          .agg(
              eco_n_linhas_transmissao=("IdeLinTms",  "nunique"),
              eco_tensao_media_kv     =("tensao",      "mean"),
          )
          .reset_index()
)
 
# --- 2c. SIGET subestacoes: substation capacity per UF ---
subs = pd.read_csv(
    RAW + "ANEEL_SIGET_subestacoes.csv",
    sep=";", encoding="latin1"
)
subs["uf"]          = norm_uf(subs["SigUFSubestacao"])
subs["pot_ativa"]   = fix_decimal(subs["MdaPotAtvMdlEqp"])
 
eco_subs = (
    subs.groupby("uf")
        .agg(eco_capacidade_subestacao_mva=("pot_ativa", "sum"))
        .reset_index()
)
 
# --- 2d. ANP Biometano: authorized capacity and idleness ---
bio_cap = pd.read_csv(
    RAW + "ANP_Biometano_capacidade.csv",
    encoding="utf-8-sig", sep=","
)
bio_cap.columns = [
    "mes_ano", "razao_social", "cnpj", "regiao", "estado",
    "municipio", "cap_autorizada_m3d", "cap_biogas_m3d",
    "vol_processado_m3d", "util_pct"
]
bio_cap["municipio"]         = norm_municipio(bio_cap["municipio"])
bio_cap["uf"]                = norm_uf(bio_cap["estado"].map(
    lambda x: x.strip() if isinstance(x, str) else x
))
bio_cap["cap_autorizada_m3d"] = fix_decimal(bio_cap["cap_autorizada_m3d"])
bio_cap["vol_processado_m3d"] = fix_decimal(bio_cap["vol_processado_m3d"])
 
eco_bio = (
    bio_cap.groupby(["municipio"])
           .agg(
               eco_bio_capacidade_m3d =("cap_autorizada_m3d", "sum"),
               eco_bio_processado_m3d =("vol_processado_m3d", "sum"),
               eco_bio_n_plantas      =("razao_social",        "nunique"),
           )
           .reset_index()
)
eco_bio["eco_bio_ociosidade"] = (
    1 - eco_bio["eco_bio_processado_m3d"] / eco_bio["eco_bio_capacidade_m3d"].replace(0, pd.NA)
).fillna(0).clip(0, 1)
 
print(f"eco_siga:  {eco_siga.shape}")
print(f"eco_linhas:{eco_linhas.shape}")
print(f"eco_subs:  {eco_subs.shape}")
print(f"eco_bio:   {eco_bio.shape}")
 
 
# ─────────────────────────────────────────────
# 3. SOCIAL VARIABLES  (prefix: soc_)
# ─────────────────────────────────────────────
 
# diversity of energy sources per municipality
soc_diversidade = (
    siga_raw.groupby(["municipio", "uf"])["tipo"]
            .nunique()
            .reset_index()
            .rename(columns={"tipo": "soc_diversidade_fontes"})
)
 
# number of operational plants (proxy for local energy economy)
soc_plantas_op = (
    siga_op.groupby(["municipio", "uf"])
           .size()
           .reset_index(name="soc_n_plantas_operacionais")
)
 
print(f"soc_diversidade:  {soc_diversidade.shape}")
print(f"soc_plantas_op:   {soc_plantas_op.shape}")
 
 
# ─────────────────────────────────────────────
# 4. ENVIRONMENTAL VARIABLES  (prefix: env_)
# ─────────────────────────────────────────────
 
# CO2 avoided (t/year) = MW fiscalizado × 8760h × fator SIN
env_co2 = (
    siga_raw[siga_raw["tipo"].isin(FONTES_RENOVAVEIS)]
    .groupby(["municipio", "uf"])["mw_fiscalizado"]
    .sum()
    .reset_index()
)
env_co2["env_co2_evitado_t_ano"] = env_co2["mw_fiscalizado"] * 8760 * FATOR_SIN
env_co2 = env_co2.drop(columns=["mw_fiscalizado"])
 
# planned transmission projects per UF (grid expansion signal)
proj = pd.read_csv(
    RAW + "ANEEL_SIGET_transmissao-projetos.csv",
    sep=";", encoding="latin1"
)
proj["uf"] = norm_uf(proj.get("SigUFSubestacaoOrigem", pd.Series([""] * len(proj))))
 
env_proj = (
    proj.groupby("DscSituacaoEpd")
        .size()
        .reset_index(name="env_n_projetos")
        .rename(columns={"DscSituacaoEpd": "situacao"})
)
# keep planned/under construction as UF-level signal
proj["uf"] = norm_uf(proj["NomMdl"].str.extract(r'\b([A-Z]{2})\b$')[0].fillna(
    proj["DscMdl"].str.extract(r'\b([A-Z]{2})\b')[0].fillna("NA")
))
env_proj_uf = (
    proj[proj["uf"] != "NA"]
        .groupby("uf")
        .agg(env_n_projetos_transmissao=("IdeObr", "nunique"))
        .reset_index()
)
 
# ANP biometano production per state (env signal: biogas utilization)
bio_prod = pd.read_csv(
    RAW + "ANP_Biometano_producao.csv",
    encoding="utf-8-sig", sep=","
)
bio_prod.columns = ["mes_ano", "regiao", "estado", "produto", "producao_m3"]
bio_prod["uf"] = norm_uf(bio_prod["estado"])
 
env_bio_prod = (
    bio_prod.groupby("uf")
            .agg(env_bio_producao_m3=("producao_m3", "sum"))
            .reset_index()
)
 
print(f"env_co2:       {env_co2.shape}")
print(f"env_proj_uf:   {env_proj_uf.shape}")
print(f"env_bio_prod:  {env_bio_prod.shape}")
 
 
# ─────────────────────────────────────────────
# 5. CONSOLIDATE INTO score_df
# ─────────────────────────────────────────────
 
score_df = core_df.copy()
 
# economic — municipality level
score_df = score_df.merge(eco_siga,  on=["municipio", "uf"], how="left")
score_df = score_df.merge(eco_bio,   on="municipio",          how="left")
 
# economic — UF level
score_df = score_df.merge(eco_linhas, on="uf", how="left")
score_df = score_df.merge(eco_subs,   on="uf", how="left")
 
# social — municipality level
score_df = score_df.merge(soc_diversidade, on=["municipio", "uf"], how="left")
score_df = score_df.merge(soc_plantas_op,  on=["municipio", "uf"], how="left")
 
# environmental — municipality level
score_df = score_df.merge(env_co2, on=["municipio", "uf"], how="left")
 
# environmental — UF level
score_df = score_df.merge(env_proj_uf,  on="uf", how="left")
score_df = score_df.merge(env_bio_prod, on="uf", how="left")
 
# ─────────────────────────────────────────────
# 6. SUMMARY
# ─────────────────────────────────────────────
 
eco_cols = [c for c in score_df.columns if c.startswith("eco_")]
soc_cols = [c for c in score_df.columns if c.startswith("soc_")]
env_cols = [c for c in score_df.columns if c.startswith("env_")]

core_df: (1938, 4)
eco_siga:  (1938, 7)
eco_linhas:(27, 3)
eco_subs:  (27, 2)
eco_bio:   (19, 5)
soc_diversidade:  (1938, 3)
soc_plantas_op:   (1816, 3)
env_co2:       (1341, 3)
env_proj_uf:   (34, 2)
env_bio_prod:  (7, 2)


/var/folders/ym/3qhchx8d19q8jp5fvb95q9vw0000gn/T/ipykernel_32955/680167167.py:88: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(0).clip(0, 1)


In [114]:
# ─────────────────────────────────────────────
# DATA DICTIONARY
# ─────────────────────────────────────────────

data_dict = {
    # index
    "municipio": {"block": "index", "source": "SIGA",  "description": "Municipality name — normalized, uppercase, no accents"},
    "uf":        {"block": "index", "source": "SIGA",  "description": "State code (2 letters)"},
    "lat":       {"block": "index", "source": "SIGA",  "description": "Latitude — mean of all plants in the municipality"},
    "lon":       {"block": "index", "source": "SIGA",  "description": "Longitude — mean of all plants in the municipality"},

    # economic
    "eco_mw_instalado":            {"block": "eco", "source": "SIGA",  "description": "Total installed generation capacity (kW)"},
    "eco_mw_outorgado":            {"block": "eco", "source": "SIGA",  "description": "Total granted generation capacity (kW)"},
    "eco_mw_ocioso":               {"block": "eco", "source": "SIGA",  "description": "Idle capacity = outorgado − instalado (kW)"},
    "eco_n_plantas":               {"block": "eco", "source": "SIGA",  "description": "Total number of generation projects"},
    "eco_share_renovavel":         {"block": "eco", "source": "SIGA",  "description": "Share of renewable MW over total installed (0–1)"},
    "eco_bio_capacidade_m3d":      {"block": "eco", "source": "ANP",   "description": "Authorized biomethane production capacity (m³/day)"},
    "eco_bio_processado_m3d":      {"block": "eco", "source": "ANP",   "description": "Actual biogas volume processed (m³/day)"},
    "eco_bio_n_plantas":           {"block": "eco", "source": "ANP",   "description": "Number of active biomethane plants"},
    "eco_bio_ociosidade":          {"block": "eco", "source": "ANP",   "description": "Biomethane idleness = 1 − (processed / authorized) (0–1)"},
    "eco_n_linhas_transmissao":    {"block": "eco", "source": "SIGET", "description": "Number of transmission lines in the state"},
    "eco_tensao_media_kv":         {"block": "eco", "source": "SIGET", "description": "Average transmission line voltage in the state (kV)"},
    "eco_capacidade_subestacao_mva":{"block": "eco", "source": "SIGET","description": "Total substation active capacity in the state (MVA)"},

    # social
    "soc_diversidade_fontes":       {"block": "soc", "source": "SIGA", "description": "Number of distinct energy source types in the municipality"},
    "soc_n_plantas_operacionais":   {"block": "soc", "source": "SIGA", "description": "Number of plants currently in operational phase"},

    # environmental
    "env_co2_evitado_t_ano":        {"block": "env", "source": "SIGA",  "description": "Estimated CO₂ avoided per year — MW renovável × 8760h × 0.06 tCO₂/MWh (t/year)"},
    "env_n_projetos_transmissao":   {"block": "env", "source": "SIGET", "description": "Number of planned transmission projects in the state"},
    "env_bio_producao_m3":          {"block": "env", "source": "ANP",   "description": "Total biomethane production in the state (m³)"},
}

# pretty print
import json
print(json.dumps(data_dict, indent=2, ensure_ascii=False))

# ─────────────────────────────────────────────
# EXPORT
# ─────────────────────────────────────────────

score_df.to_csv("../data/processed/score_df.csv", index=False, encoding="utf-8-sig")

{
  "municipio": {
    "block": "index",
    "source": "SIGA",
    "description": "Municipality name — normalized, uppercase, no accents"
  },
  "uf": {
    "block": "index",
    "source": "SIGA",
    "description": "State code (2 letters)"
  },
  "lat": {
    "block": "index",
    "source": "SIGA",
    "description": "Latitude — mean of all plants in the municipality"
  },
  "lon": {
    "block": "index",
    "source": "SIGA",
    "description": "Longitude — mean of all plants in the municipality"
  },
  "eco_mw_instalado": {
    "block": "eco",
    "source": "SIGA",
    "description": "Total installed generation capacity (kW)"
  },
  "eco_mw_outorgado": {
    "block": "eco",
    "source": "SIGA",
    "description": "Total granted generation capacity (kW)"
  },
  "eco_mw_ocioso": {
    "block": "eco",
    "source": "SIGA",
    "description": "Idle capacity = outorgado − instalado (kW)"
  },
  "eco_n_plantas": {
    "block": "eco",
    "source": "SIGA",
    "description": "Total number

### Modeling

In [115]:
FONTES = ["Solar", "Eolica", "Biometano", "H2 Verde", "Biomassa"]

VAR_FONTE = {
    "Solar": {
        "eco": ["eco_mw_instalado", "eco_share_renovavel", "eco_n_linhas_transmissao"],
        "soc": ["soc_diversidade_fontes", "soc_n_plantas_operacionais"],
        "env": ["env_co2_evitado_t_ano", "env_n_projetos_transmissao"],
    },
    "Eolica": {
        "eco": ["eco_mw_instalado", "eco_mw_ocioso", "eco_n_linhas_transmissao"],
        "soc": ["soc_diversidade_fontes", "soc_n_plantas_operacionais"],
        "env": ["env_co2_evitado_t_ano", "env_n_projetos_transmissao"],
    },
    "Biometano": {
        "eco": ["eco_bio_capacidade_m3d", "eco_bio_ociosidade", "eco_n_linhas_transmissao"],
        "soc": ["soc_diversidade_fontes", "soc_n_plantas_operacionais"],
        "env": ["env_co2_evitado_t_ano", "env_bio_producao_m3"],
    },
    "H2 Verde": {
        "eco": ["eco_mw_instalado", "eco_share_renovavel", "eco_capacidade_subestacao_mva"],
        "soc": ["soc_diversidade_fontes", "soc_n_plantas_operacionais"],
        "env": ["env_co2_evitado_t_ano", "env_n_projetos_transmissao"],
    },
    "Biomassa": {
        "eco": ["eco_mw_instalado", "eco_n_linhas_transmissao", "eco_capacidade_subestacao_mva"],
        "soc": ["soc_diversidade_fontes", "soc_n_plantas_operacionais"],
        "env": ["env_co2_evitado_t_ano", "env_bio_producao_m3"],
    },
}

# weights
W_ECO, W_SOC, W_ENV = 0.40, 0.30, 0.30

# ── normalize all variables globally ──────────
def minmax(s):
    mn, mx = s.min(), s.max()
    return pd.Series([0.5]*len(s), index=s.index) if mx == mn else (s - mn) / (mx - mn)

all_vars = list({v for f in VAR_FONTE.values() for blk in f.values() for v in blk})

norm = score_df[["municipio", "uf", "lat", "lon"]].copy()
for v in all_vars:
    col = score_df[v].fillna(0) if v in score_df.columns else pd.Series(0, index=score_df.index)
    norm[v + "_n"] = minmax(col)

# ── compute score per municipality × fonte ────
rows = []
for _, row in norm.iterrows():
    for fonte in FONTES:
        blks  = VAR_FONTE[fonte]
        eco_s = np.mean([row.get(v + "_n", 0) for v in blks["eco"]])
        soc_s = np.mean([row.get(v + "_n", 0) for v in blks["soc"]])
        env_s = np.mean([row.get(v + "_n", 0) for v in blks["env"]])
        total = W_ECO * eco_s + W_SOC * soc_s + W_ENV * env_s
        rows.append({
            "municipio": row["municipio"],
            "uf":        row["uf"],
            "lat":       row["lat"],
            "lon":       row["lon"],
            "fonte":     fonte,
            "score":     round(total, 4),
            "eco_score": round(eco_s, 4),
            "soc_score": round(soc_s, 4),
            "env_score": round(env_s, 4),
        })

mcda_df = pd.DataFrame(rows)

# ── rank fontes within each municipality ──────
mcda_df["rank"] = (
    mcda_df.groupby("municipio")["score"]
           .rank(method="dense", ascending=False)
           .astype(int)
)

# ── best fit per municipality (rank 1 only) ───
best_df = (
    mcda_df[mcda_df["rank"] == 1]
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)

In [116]:
mcda_df["rank"] = (
    mcda_df.groupby("municipio")["score"]
           .rank(method="dense", ascending=False)
           .astype(int)
)

ranking_df = (
    mcda_df.sort_values(["municipio", "rank"])
           .reset_index(drop=True)
)


### Datasets

In [117]:
ranking_df.to_csv("../data/processed/best_energy_ranking.csv", index=False, sep=",", decimal=".")
best_df.to_csv("../data/processed/score_per_set_variable.csv",       index=False, sep=",", decimal=".")
score_df.to_csv("../data/processed/descriptive_dataset.csv",     index=False, sep=",", decimal=".")
